Objective: 
The goal of this assignment is to provide hands-on experience with the end-to-end process 
of collecting domain-specific data, building a dataset, and pretraining a GPT-2 
language model from scratch on the collected dataset. 
You will practice data acquisition, preprocessing, and model training workflows critical to 
language model development. 

Part 1: Data Collection 
• Country Selection: Choose one country (e.g., Australia, Canada, Pakistan, etc.). 
• Sources: Collect text data only from official government or educational 
institution websites (e.g., .gov, .edu domains). 
• Language: Data must be in English. 
• Quantity Target: Aim for at least 1 million words (~5-10 MB of clean text). 
• Allowed Content: Policies, reports, educational materials, public announcements, 
manuals, etc. 
• Forbidden Content: News articles from non-government sources, blogs, private
sector websites, or copyrighted materials not belonging to the public domain. 
Deliverables: 
• Part 1 report with the following details: 
o Country selected 
o List of URLs where data was collected 
o Description (1-2 sentences per URL) explaining the nature of content 
• The final cleaned dataset in a single .txt file (one paragraph per line, minimal noise) 

In [7]:
!pip install bs4

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
  Using cached beautifulsoup4-4.13.4-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.7-py3-none-any.whl.metadata (4.6 kB)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
Using cached beautifulsoup4-4.13.4-py3-none-any.whl (187 kB)
Using cached soupsieve-2.7-py3-none-any.whl (36 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
!pip install PyPDF2

In [9]:
!pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 645.7 kB/s eta 0:00:03
   ----------------- ---------------------- 0.8/1.8 MB 907.1 kB/s eta 0:00:02
   ---------------------- ----------------- 1.0/1.8 MB 1.1 MB/s eta 0:00:01
   ---------------------- ----------------- 1.0/1.8 MB 1.1 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 1.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1


In [ ]:
# --- 1. Import Necessary Libraries ---
import requests
from bs4 import BeautifulSoup
import time
import re
import logging
from urllib.parse import urljoin, urlparse
import os
from collections import deque 
import PyPDF2 
import io 
import shutil 

Configuration

In [ ]:
# --- 2. Configuration Settings ---

# --- Core Parameters ---
COUNTRY_NAME = "New Zealand" 
SEED_URLS = [
    "https://www.govt.nz/",
    "https://www.education.govt.nz/",
    "https://www.health.govt.nz/",
    "https://www.mbie.govt.nz/",
    "https://www.mpi.govt.nz/",
    "https://www.stats.govt.nz/",
    "https://www.auckland.ac.nz/en.html",
    "https://www.canterbury.ac.nz/",
    "https://www.otago.ac.nz/",
]
ALLOWED_DOMAINS_SUFFIXES = ('.govt.nz', '.ac.nz', '.mil.nz', '.parliament.nz')

# --- Quantity & Limits ---
TARGET_WORD_COUNT_OVERALL = 1_000_000 # Global target across all sites
MAX_PAGES_PER_SITE = 500 # Safety limit per individual site crawl - ADJUST AS NEEDED
MIN_PARAGRAPH_LENGTH = 25 # Min chars for a text block to be kept

# --- Output ---
TEMP_OUTPUT_DIR = "temp_scraped_data" # Directory to store individual site files
FINAL_OUTPUT_FILE = "single.txt"
OUTPUT_REPORT_FILE = f"{COUNTRY_NAME.lower().replace(' ','_')}_report_data.txt" # Raw data for report

# --- Request Settings ---
REQUEST_DELAY_SECONDS = 3 # Increase delay for deeper crawling
REQUEST_TIMEOUT_SECONDS = 30
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9', # Prefer English
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8,application/pdf;q=0.7' # Accept HTML and PDF
}

In [26]:
# --- 3. Setup Logging ---
log_format = '%(asctime)s - %(levelname)s - %(message)s'
logging.basicConfig(level=logging.INFO, format=log_format)
# Optional: Log to file
# logging.getLogger().addHandler(logging.FileHandler("crawler.log"))

logging.info(f"Logging configured for {COUNTRY_NAME} data collection.")

2025-05-03 21:42:23,701 - INFO - Logging configured for New Zealand data collection.


In [27]:
# --- 4. Helper Functions ---

def get_netloc(url):
    """Extracts the network location (domain name/subdomain) from a URL."""
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return None

def is_valid_initial_seed(url, allowed_suffixes):
    """Checks if a URL is a valid starting point based on domain suffix."""
    try:
        domain = urlparse(url).netloc.lower()
        return domain.endswith(allowed_suffixes)
    except Exception:
        return False

def is_valid_crawl_target_within_site(url, site_netloc):
    """Checks if the URL belongs to the target site and isn't an obvious non-content file."""
    try:
        parsed_url = urlparse(url)
        current_netloc = parsed_url.netloc.lower()
        path = parsed_url.path.lower()

        # Must belong to the same site (e.g., www.govt.nz)
        if current_netloc != site_netloc:
            return False

        # Avoid common non-content file types (add more if needed)
        excluded_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.svg', '.ico',
                               '.css', '.js',
                               '.zip', '.rar', '.exe', '.dmg', '.tar', '.gz',
                               '.mp3', '.mp4', '.avi', '.mov', '.wmv',
                               '.xls', '.xlsx', '.doc', '.docx', '.ppt', '.pptx') # Keep .pdf!
        if path.endswith(excluded_extensions):
            return False

        # Avoid mailto links
        if parsed_url.scheme == 'mailto':
            return False

        return True

    except Exception as e:
        logging.warning(f"URL validation error for {url}: {e}")
        return False

def extract_title(html_content):
    """Extracts the <title> tag content from HTML."""
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        title_tag = soup.find('title')
        if title_tag and title_tag.string:
            return title_tag.string.strip()
    except Exception as e:
        logging.warning(f"Error extracting title: {e}")
    return "No Title Found"

def count_words(text):
    """Estimates the number of words in a string."""
    return len(text.split())

def sanitize_filename(name):
    """Removes characters unsuitable for filenames."""
    # Remove http(s)://
    name = re.sub(r'^https?:\/\/', '', name)
    # Replace dots and other invalid chars with underscores
    name = re.sub(r'[^\w\-.]+', '_', name)
    # Remove leading/trailing underscores
    name = name.strip('_')
    return name

In [29]:
# --- 5. Content Fetching Functions ---

def fetch_content(url, headers, timeout):
    """Fetches content (HTML or PDF), returning response object or None."""
    logging.info(f"Attempting to fetch: {url}")
    try:
        response = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True, stream=True) # Use stream for PDFs
        response.raise_for_status()
        logging.info(f"Successfully fetched headers for {url} (Status: {response.status_code}, Type: {response.headers.get('Content-Type')})")
        return response
    except requests.exceptions.Timeout:
        logging.error(f"Request timed out for {url} after {timeout} seconds.")
        return None
    except requests.exceptions.RequestException as e:
        # Log common errors like connection refused, 404 etc.
        logging.error(f"Request failed for {url}: {e}")
        return None
    except Exception as e:
        logging.error(f"An unexpected error occurred while fetching {url}: {e}")
        return None

def extract_text_from_pdf(pdf_content_bytes):
    """Extracts text from PDF byte content using PyPDF2."""
    text = ""
    try:
        pdf_file = io.BytesIO(pdf_content_bytes)
        reader = PyPDF2.PdfReader(pdf_file)
        num_pages = len(reader.pages)
        for page_num in range(num_pages):
            page = reader.pages[page_num]
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n" # Add newline between pages
        logging.info(f"Extracted text from PDF ({num_pages} pages)")
        return text
    except PyPDF2.errors.PdfReadError as e:
        logging.error(f"PyPDF2 could not read PDF: {e}")
        return None
    except Exception as e:
        logging.error(f"Error processing PDF content: {e}")
        return None

In [30]:
# --- 6. HTML Parsing, Link Extraction, Text Cleaning ---

def extract_links(html_content, base_url, site_netloc):
    """Extracts absolute links from HTML content that belong to the same site_netloc."""
    links = set()
    try:
        soup = BeautifulSoup(html_content, 'html.parser')
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            if href and not href.startswith('#') and not href.startswith('javascript:'):
                absolute_url = urljoin(base_url, href)
                # Clean URL (remove fragment)
                absolute_url_parsed = urlparse(absolute_url)
                absolute_url_cleaned = absolute_url_parsed._replace(fragment="").geturl()

                # IMPORTANT: Check if the link belongs to the *same site* we are currently crawling
                if get_netloc(absolute_url_cleaned) == site_netloc:
                    links.add(absolute_url_cleaned)

        logging.debug(f"Found {len(links)} potential links matching site {site_netloc}.")
        return links
    except Exception as e:
        logging.warning(f"Error extracting links from {base_url}: {e}")
        return set()


def extract_and_clean_text_from_html(html_content, min_para_length):
    """Parses HTML, extracts text blocks, cleans them, and returns a list of paragraphs."""
    paragraphs = []
    if not html_content:
        return paragraphs

    soup = BeautifulSoup(html_content, 'html.parser')

    # Basic noise removal
    for element in soup(["script", "style", "header", "footer", "nav", "aside", "form", "button", "img", "noscript", "figure", "figcaption", "iframe", "svg"]):
        element.decompose()

    # Target main content areas (add more selectors if needed)
    main_content = soup.find('main') or \
                   soup.find('article') or \
                   soup.find('div', id=re.compile(r'content|main|body|container|page', re.I)) or \
                   soup.find('div', class_=re.compile(r'content|main|body|entry|article|page', re.I))
    target_element = main_content if main_content else soup.body

    if not target_element:
        logging.warning("Could not find target element (main/body) for text extraction.")
        return paragraphs

    # Extract text using stripped_strings for cleaner segments
    for string in target_element.stripped_strings:
        cleaned_text = re.sub(r'\s+', ' ', string).strip() # Normalize whitespace

        # Simple check for English-like content (avoids large blocks of symbols/code)
        # This is NOT a robust language check.
        if cleaned_text and re.match(r'^[a-zA-Z0-9\s.,!?;:\'"()\[\]%&$£€+-/]+$', cleaned_text):
             if len(cleaned_text) >= min_para_length:
                # Add more cleaning rules here if needed (e.g., remove specific boilerplate)
                paragraphs.append(cleaned_text)

    logging.debug(f"Extracted {len(paragraphs)} cleaned paragraphs from HTML.")
    return paragraphs

def clean_pdf_text(raw_text, min_para_length):
    """Cleans text extracted from PDFs, splitting into paragraphs."""
    paragraphs = []
    if not raw_text:
        return paragraphs

    # Split text into potential paragraphs (e.g., by double newlines or just single)
    lines = raw_text.split('\n')
    current_paragraph = ""

    for line in lines:
        cleaned_line = re.sub(r'\s+', ' ', line).strip()
        if cleaned_line:
            current_paragraph += cleaned_line + " " # Combine lines into paragraphs
        elif current_paragraph.strip(): # End of a paragraph block (empty line)
            final_paragraph = current_paragraph.strip()
            if len(final_paragraph) >= min_para_length:
                 paragraphs.append(final_paragraph)
            current_paragraph = ""

    # Add the last paragraph if it wasn't followed by an empty line
    final_paragraph = current_paragraph.strip()
    if len(final_paragraph) >= min_para_length:
        paragraphs.append(final_paragraph)

    logging.debug(f"Processed PDF text into {len(paragraphs)} paragraphs.")
    return paragraphs

In [31]:
# --- 7. Site-by-Site Crawler Logic ---

# Ensure the temporary output directory exists
if not os.path.exists(TEMP_OUTPUT_DIR):
    os.makedirs(TEMP_OUTPUT_DIR)
    logging.info(f"Created temporary directory: {TEMP_OUTPUT_DIR}")

# --- Global Tracking ---
overall_total_word_count = 0
scraped_sources_report = {} # Store {url: title} for reporting across all sites
processed_site_files = [] # Keep track of generated files for merging

logging.info(f"Starting site-by-site crawl. Overall Target: {TARGET_WORD_COUNT_OVERALL:,} words.")
logging.info(f"Max pages per site: {MAX_PAGES_PER_SITE}. Allowed suffixes: {ALLOWED_DOMAINS_SUFFIXES}")

# --- Loop through each seed URL ---
for seed_url in SEED_URLS:

    # Check if overall word count target is already met
    if overall_total_word_count >= TARGET_WORD_COUNT_OVERALL:
        logging.info(f"Overall word target ({TARGET_WORD_COUNT_OVERALL:,}) reached. Stopping crawl.")
        break

    # Validate the seed URL itself
    if not is_valid_initial_seed(seed_url, ALLOWED_DOMAINS_SUFFIXES):
        logging.warning(f"Skipping invalid seed URL: {seed_url}")
        continue

    site_netloc = get_netloc(seed_url)
    if not site_netloc:
        logging.warning(f"Could not determine network location for seed: {seed_url}. Skipping.")
        continue

    logging.info(f"\n{'='*20} Starting crawl for site: {site_netloc} {'='*20}")

    # --- Site-Specific Tracking ---
    site_urls_to_visit = deque([seed_url])
    site_visited_urls = set()
    site_paragraphs = set() # Use set for deduplication within the site
    site_pages_crawled = 0
    site_word_count = 0

    # --- Crawl Loop for the Current Site ---
    while site_urls_to_visit and site_pages_crawled < MAX_PAGES_PER_SITE and overall_total_word_count < TARGET_WORD_COUNT_OVERALL:
        current_url = site_urls_to_visit.popleft()

        # 1. Check if already visited (within this site crawl) or invalid target
        if current_url in site_visited_urls:
            logging.debug(f"[{site_netloc}] Skipping already visited: {current_url}")
            continue

        # Check if the URL still belongs to the target site (paranoia check)
        if not is_valid_crawl_target_within_site(current_url, site_netloc):
            logging.debug(f"[{site_netloc}] Skipping invalid target for this site: {current_url}")
            site_visited_urls.add(current_url) # Add invalid targets to visited to avoid re-checking
            continue

        # Add to visited BEFORE fetching
        site_visited_urls.add(current_url)
        site_pages_crawled += 1

        # 2. Fetch content
        response = fetch_content(current_url, HEADERS, REQUEST_TIMEOUT_SECONDS)

        if not response:
            logging.warning(f"[{site_netloc}] Fetch failed for {current_url}, skipping.")
            time.sleep(REQUEST_DELAY_SECONDS / 2)
            continue

        content_type = response.headers.get('Content-Type', '').lower()
        page_title = "N/A (PDF or Title Error)"
        new_paragraphs = []
        extracted_links = set()

        try:
            # 3. Process based on content type
            if 'application/pdf' in content_type:
                logging.info(f"[{site_netloc}] Processing PDF: {current_url}")
                pdf_bytes = response.content
                raw_pdf_text = extract_text_from_pdf(pdf_bytes)
                if raw_pdf_text:
                    new_paragraphs = clean_pdf_text(raw_pdf_text, MIN_PARAGRAPH_LENGTH)
                    page_title = os.path.basename(urlparse(current_url).path) # Use filename

            elif 'text/html' in content_type:
                logging.info(f"[{site_netloc}] Processing HTML: {current_url}")
                html_content = response.text
                page_title = extract_title(html_content)
                new_paragraphs = extract_and_clean_text_from_html(html_content, MIN_PARAGRAPH_LENGTH)
                # Pass site_netloc to ensure links stay within the site
                extracted_links = extract_links(html_content, current_url, site_netloc)

            else:
                logging.warning(f"[{site_netloc}] Skipping unsupported content type '{content_type}' at: {current_url}")

            # 4. Store results if paragraphs found
            if new_paragraphs:
                words_added_this_page = 0
                paragraphs_added_this_page = 0
                for para in new_paragraphs:
                    if para not in site_paragraphs:
                        site_paragraphs.add(para)
                        para_words = count_words(para)
                        site_word_count += para_words
                        overall_total_word_count += para_words # Increment global count
                        words_added_this_page += para_words
                        paragraphs_added_this_page += 1

                if paragraphs_added_this_page > 0:
                    scraped_sources_report[current_url] = page_title # Add to global report list
                    logging.info(f"[{site_netloc}] Added {paragraphs_added_this_page} new unique paragraphs ({words_added_this_page} words) from {current_url}. Site words: {site_word_count:,}. Overall words: {overall_total_word_count:,}")

                    # Check if overall target met after adding paragraphs
                    if overall_total_word_count >= TARGET_WORD_COUNT_OVERALL:
                        logging.info(f"Overall word target ({TARGET_WORD_COUNT_OVERALL:,}) reached during processing {current_url}. Stopping crawl.")
                        # Need to break out of inner and outer loops - will be handled by the check at start of loops
            else:
                logging.info(f"[{site_netloc}] No new paragraphs extracted or passed cleaning for {current_url}")

            # 5. Add new valid links to the site-specific queue
            for link in extracted_links:
                # Check if visited within this site's crawl
                if link not in site_visited_urls and link not in site_urls_to_visit:
                    # is_valid_crawl_target_within_site already checked in extract_links implicitly
                     site_urls_to_visit.append(link)
                     logging.debug(f"[{site_netloc}] Added to queue: {link}")

        except Exception as e:
            logging.error(f"[{site_netloc}] Error processing content from {current_url}: {e}", exc_info=True)

        finally:
            if response:
                response.close()

        # 6. Polite Delay
        logging.debug(f"[{site_netloc}] Queue: {len(site_urls_to_visit)}. Visited: {len(site_visited_urls)}. Crawled: {site_pages_crawled}/{MAX_PAGES_PER_SITE}. Site Words: {site_word_count}. Overall Words: {overall_total_word_count}")
        time.sleep(REQUEST_DELAY_SECONDS)

    # --- End of Crawl Loop for the Site ---
    logging.info(f"Finished crawl for site: {site_netloc}. Pages crawled: {site_pages_crawled}. Paragraphs collected: {len(site_paragraphs):,}. Words collected: {site_word_count:,}")
    if site_pages_crawled >= MAX_PAGES_PER_SITE:
        logging.warning(f"[{site_netloc}] Stopped crawl due to reaching MAX_PAGES_PER_SITE limit ({MAX_PAGES_PER_SITE}).")
    if not site_urls_to_visit:
        logging.info(f"[{site_netloc}] Stopped crawl because the URL queue for this site is empty.")

    # --- Save paragraphs for this site ---
    if site_paragraphs:
        site_filename_base = sanitize_filename(site_netloc)
        site_output_file = os.path.join(TEMP_OUTPUT_DIR, f"{site_filename_base}.txt")
        try:
            with open(site_output_file, 'w', encoding='utf-8') as f:
                sorted_paragraphs = sorted(list(site_paragraphs))
                for para in sorted_paragraphs:
                    f.write(para + '\n')
            logging.info(f"Saved {len(site_paragraphs)} paragraphs for {site_netloc} to {site_output_file}")
            processed_site_files.append(site_output_file) # Add to list for merging
        except IOError as e:
            logging.error(f"Failed to write site file {site_output_file}: {e}")
    else:
        logging.info(f"No paragraphs collected for site {site_netloc}, skipping file save.")


# --- End of Loop through Seed URLs ---
logging.info("\n" + "="*20 + " ALL SITE CRAWLS FINISHED " + "="*20)
logging.info(f"Total unique sources across all sites: {len(scraped_sources_report):,}")
logging.info(f"Estimated total words collected overall: {overall_total_word_count:,}")
logging.info(f"Individual site files saved in: {TEMP_OUTPUT_DIR}")

print(f"\nSite-by-site crawling finished. Collected approximately {overall_total_word_count:,} words.")
print(f"Found sources from {len(scraped_sources_report):,} URLs across all processed sites.")

2025-05-03 21:43:15,735 - INFO - Created temporary directory: temp_scraped_data
2025-05-03 21:43:15,735 - INFO - Starting site-by-site crawl. Overall Target: 1,000,000 words.
2025-05-03 21:43:15,735 - INFO - Max pages per site: 500. Allowed suffixes: ('.govt.nz', '.ac.nz', '.mil.nz', '.parliament.nz')
2025-05-03 21:43:15,743 - INFO - 
==================== Starting crawl for site: www.govt.nz ====================
2025-05-03 21:43:15,743 - INFO - Attempting to fetch: https://www.govt.nz/
2025-05-03 21:43:17,447 - INFO - Successfully fetched headers for https://www.govt.nz/ (Status: 200, Type: text/html; charset=utf-8)
2025-05-03 21:43:17,449 - INFO - [www.govt.nz] Processing HTML: https://www.govt.nz/
2025-05-03 21:43:17,713 - INFO - [www.govt.nz] Added 34 new unique paragraphs (286 words) from https://www.govt.nz/. Site words: 286. Overall words: 286
2025-05-03 21:43:20,718 - INFO - Attempting to fetch: https://www.govt.nz/browse/history-culture-and-heritage/
2025-05-03 21:43:22,233 - I

KeyboardInterrupt: 

In [ ]:
# --- 8. Merge Individual Site Files and Save Report ---

logging.info(f"Merging {len(processed_site_files)} site files into {FINAL_OUTPUT_FILE}...")

try:
    with open(FINAL_OUTPUT_FILE, 'wb') as outfile: # Open in binary write mode
        for filename in processed_site_files:
            if os.path.exists(filename):
                try:
                    with open(filename, 'rb') as infile: # Open in binary read mode
                        shutil.copyfileobj(infile, outfile)
                    logging.debug(f"Appended {filename} to {FINAL_OUTPUT_FILE}")
                except IOError as e:
                    logging.error(f"Error reading site file {filename} during merge: {e}")
            else:
                logging.warning(f"Site file {filename} not found during merge, skipping.")
    logging.info(f"Successfully merged files into {FINAL_OUTPUT_FILE}")
    print(f"Final combined dataset saved to '{FINAL_OUTPUT_FILE}'")

    # Optional: Clean up temporary directory after successful merge
    # try:
    #     shutil.rmtree(TEMP_OUTPUT_DIR)
    #     logging.info(f"Removed temporary directory: {TEMP_OUTPUT_DIR}")
    # except OSError as e:
    #     logging.error(f"Error removing temporary directory {TEMP_OUTPUT_DIR}: {e}")

except IOError as e:
    logging.error(f"Failed to open or write final output file {FINAL_OUTPUT_FILE}: {e}")
    print(f"ERROR: Failed to create final merged file '{FINAL_OUTPUT_FILE}'")
except Exception as e:
     logging.error(f"An unexpected error occurred during merging: {e}")
     print(f"ERROR: An unexpected error occurred during merging.")


# --- Save the final report data (URL and Title) ---
logging.info(f"Saving report data to {OUTPUT_REPORT_FILE}")
try:
    with open(OUTPUT_REPORT_FILE, 'w', encoding='utf-8') as f:
        f.write(f"# Report Data for: {COUNTRY_NAME}\n")
        f.write(f"# Total Sources Included: {len(scraped_sources_report)}\n")
        f.write("# Format: URL | Extracted Title (requires manual description)\n")
        f.write("="*30 + "\n")
        # Sort report by URL for consistency
        for url, title in sorted(scraped_sources_report.items()):
             cleaned_title = title.replace('\n', ' ').replace('\r', '').strip()
             f.write(f"{url} | {cleaned_title}\n")
    logging.info(f"Successfully saved report data.")
    print(f"Report data (URL list) saved to '{OUTPUT_REPORT_FILE}'. You will need to add descriptions.")
except IOError as e:
    logging.error(f"Failed to write report data: {e}")
    print(f"ERROR: Failed to save report data to '{OUTPUT_REPORT_FILE}'")